# Beyond Persuasion - Presentation Demo

This notebook is intentionally simple and presentation-oriented.

Its goal is to show the code pipeline implemented:

`ConversationTurn -> EmotionAnalyzer -> EthicalEngine -> Prompting -> LocalLLMClient -> SafeConversationAgent`

The notebook uses the real local GGUF backend by default. The unguarded baseline uses a controlled `commercial` prompt profile, so the exam demo can show the substantial behavioral difference between a conversion-oriented assistant and the protected ethical guardrail.

## 1. Setup and imports

This cell checks that the notebook dependencies are available, locates the project root, and imports the main classes directly from the repository code.

If something is missing, the error message tells you exactly what to install.

In [ ]:
from pathlib import Path
import sys
import textwrap

MISSING_PACKAGES = []

try:
    import pandas as pd
except ImportError:
    MISSING_PACKAGES.append("pandas")

try:
    import matplotlib.pyplot as plt
except ImportError:
    MISSING_PACKAGES.append("matplotlib")

try:
    from IPython.display import Markdown, display
except ImportError:
    MISSING_PACKAGES.append("ipython")

if MISSING_PACKAGES:
    missing_text = ", ".join(MISSING_PACKAGES)
    raise ImportError(
        f"Missing notebook dependencies: {missing_text}.\n"
        "Install them inside the project environment with:\n"
        "  uv pip install -e '.[presentation]'\n"
        "Then select the .venv kernel in Jupyter or VS Code."
    )

pd.set_option("display.max_colwidth", None)


def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path] + list(start_path.parents):
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from beyond_persuasion.affective.pipeline import EmotionAnalyzer, EmotionAnalyzerConfig
from beyond_persuasion.ethics.engine import EthicalEngine
from beyond_persuasion.ethics.rules import EthicalRulesConfig
from beyond_persuasion.evaluation.runner import EvaluationRunner, EvaluationRunnerConfig
from beyond_persuasion.llm.local_model import LocalLLMClient, LocalLLMConfig
from beyond_persuasion.llm.prompting import build_system_prompt, build_user_prompt
from beyond_persuasion.orchestration.agent import SafeConversationAgent
from beyond_persuasion.schemas import ConversationTurn, EthicalAssessment

print(f"Project root: {PROJECT_ROOT}")

## 2. Load a few significant prompts from the presentation dataset

The repository includes a small presentation-oriented subset in `data/evaluation/presentation_prompts.csv`.

These prompts were chosen to remain emotionally meaningful while avoiding very strong crisis or safety-trigger wording that can dominate the model response.

For the oral demo, we use a few representative ones:
- vulnerability + purchase susceptibility,
- breakup-related suggestibility,
- financial stress,
- a neutral baseline.


In [ ]:
DATASET_PATH = PROJECT_ROOT / "data" / "evaluation" / "presentation_prompts.csv"
examples_df = pd.read_csv(DATASET_PATH)

selected_ids = [
    "demo_purchase_01",
    "demo_purchase_02",
    "demo_stress_01",
    "demo_neutral_01",
]

demo_examples_df = examples_df.loc[examples_df["example_id"].isin(selected_ids)].copy()
demo_examples_df = demo_examples_df.reset_index(drop=True)
display(demo_examples_df[["example_id", "expected_condition", "notes", "prompt_text"]])


## 3. Instantiate the core classes

Here I create the exact classes used by the project:
- `EmotionAnalyzer`
- `EthicalEngine`
- `LocalLLMClient`
- `SafeConversationAgent`


In [ ]:
GGUF_PATH = PROJECT_ROOT / "models" / "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"
TRANSFORMER_MODEL = "SamLowe/roberta-base-go_emotions"
BASELINE_PROMPT_PROFILE = "commercial"


def build_llm_config() -> LocalLLMConfig:
    if not GGUF_PATH.exists():
        raise FileNotFoundError(
            f"GGUF model not found at {GGUF_PATH}. "
            "Place the local model there before running the live demo."
        )

    try:
        import llama_cpp
    except Exception as exc:
        raise ImportError(
            "llama-cpp-python is required for the real local LLM demo. "
            "Install the optional LLM dependencies with: uv pip install -e '.[llm]'"
        ) from exc

    return LocalLLMConfig(
        backend="llama_cpp",
        model_path=str(GGUF_PATH),
        chat_format=None,
        max_tokens=128,
        temperature=0.0,
        n_ctx=2048,
        n_gpu_layers=-1,
    )


emotion_config = EmotionAnalyzerConfig(
    model_name=TRANSFORMER_MODEL,
    use_transformers=True,
)
emotion_analyzer = EmotionAnalyzer(emotion_config)

ethics_config = EthicalRulesConfig(
    primary_emotion_threshold=0.60,
    combined_emotion_threshold=0.70,
    minimum_confidence_for_flag=0.55,
)
ethical_engine = EthicalEngine(ethics_config)

llm_config = build_llm_config()
llm_client = LocalLLMClient(llm_config)

agent = SafeConversationAgent(
    emotion_analyzer=emotion_analyzer,
    ethical_engine=ethical_engine,
    llm_client=llm_client,
)

print("Affective model:", emotion_config.model_name)
print("LLM backend:", llm_config.backend)
print("Baseline prompt profile:", BASELINE_PROMPT_PROFILE)

## 4. Create a `ConversationTurn`

The pipeline starts from a `ConversationTurn` object. This is the first project-specific data structure that carries the user message through the system.

In [ ]:
demo_row = examples_df.loc[examples_df["example_id"] == "demo_purchase_01"].iloc[0]

turn = ConversationTurn(
    user_text=demo_row["prompt_text"],
    metadata={"example_id": demo_row["example_id"], "notes": demo_row["notes"]},
)

turn

## 5. Run the affective layer

Now we call `EmotionAnalyzer.predict(...)` directly.

This produces an `EmotionPrediction` object with:
- a dominant label,
- a confidence value,
- a score distribution over the project emotions.

In [ ]:
prediction = emotion_analyzer.predict(turn.user_text)

prediction_summary_df = pd.DataFrame([
    {
        "label": prediction.label,
        "confidence": round(prediction.confidence, 4),
        "scores": prediction.scores,
    }
])
display(prediction_summary_df)

score_series = pd.Series(prediction.scores).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(score_series.index, score_series.values, color=["#d95f02", "#7570b3", "#1b9e77", "#e7298a", "#66a61e"])
ax.set_title("EmotionPrediction scores")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.show()

## 6. Run the ethical layer

Here we pass the `EmotionPrediction` into the `EthicalEngine`.

This is the key transition from NLP signal to ethical decision. The result is an `EthicalAssessment` with:
- vulnerability flag,
- risk score,
- triggered rules,
- rationale.

In [ ]:
assessment = ethical_engine.assess(prediction)

assessment_df = pd.DataFrame([
    {
        "is_vulnerable": assessment.is_vulnerable,
        "risk_score": round(assessment.risk_score, 4),
        "triggered_rules": ", ".join(assessment.triggered_rules),
        "rationale": assessment.rationale,
    }
])
display(assessment_df)

## 7. Build the prompts

This step uses the functions in `llm/prompting.py`.

We show three prompt variants:
- `standard`: ordinary helpful assistant
- `commercial`: unguarded baseline that simulates a persuasive commercial deployment
- `protected`: automatically selected when the ethical engine flags vulnerability

In [ ]:
neutral_assessment = EthicalAssessment(
    is_vulnerable=False,
    risk_score=assessment.risk_score,
    rationale="Notebook baseline without protection.",
    triggered_rules=[],
)

standard_system_prompt = build_system_prompt(neutral_assessment, prompt_profile="standard")
commercial_system_prompt = build_system_prompt(neutral_assessment, prompt_profile=BASELINE_PROMPT_PROFILE)
protected_system_prompt = build_system_prompt(assessment, prompt_profile="standard")
user_prompt = build_user_prompt(turn)

print("STANDARD SYSTEM PROMPT")
print("-" * 80)
print(standard_system_prompt)
print()

print("COMMERCIAL SYSTEM PROMPT")
print("-" * 80)
print(commercial_system_prompt)
print()

print("PROTECTED SYSTEM PROMPT")
print("-" * 80)
print(protected_system_prompt)
print()

print("USER PROMPT")
print("-" * 80)
print(user_prompt)

## 8. Generate responses with `LocalLLMClient`

Now we call the local LLM layer directly.

This cell is the key demonstration: the same real Llama model receives the same user message, but different system prompts. The commercial baseline represents the risky unprotected behavior; the protected prompt represents the ethical intervention.

In [ ]:
standard_response = llm_client.generate(
    system_prompt=standard_system_prompt,
    user_prompt=user_prompt,
)
commercial_response = llm_client.generate(
    system_prompt=commercial_system_prompt,
    user_prompt=user_prompt,
)
protected_response = llm_client.generate(
    system_prompt=protected_system_prompt,
    user_prompt=user_prompt,
)

response_df = pd.DataFrame([
    {"profile": "standard", "response": standard_response},
    {"profile": BASELINE_PROMPT_PROFILE, "response": commercial_response},
    {"profile": "protected", "response": protected_response},
])
display(response_df)

## 9. Run the full pipeline with `SafeConversationAgent`

The previous cells showed the components step by step.

Now we use the high-level orchestrator, `SafeConversationAgent`, which internally calls the same components in sequence.

In [ ]:
agent_result = agent.run(turn)

agent_result_df = pd.DataFrame([
    {
        "prediction_label": agent_result.prediction.label,
        "risk_score": round(agent_result.assessment.risk_score, 4),
        "protection_enabled": agent_result.assessment.protection_enabled,
        "response_text": agent_result.response_text,
    }
])
display(agent_result_df)

## 10. Small batch comparison with `EvaluationRunner`

To conclude the live demo, we run a small guarded vs unguarded evaluation on a few significant prompts.

Here the baseline is explicitly `commercial`, so the unguarded side represents the risky behavior that the project is designed to reduce.

In [ ]:
presentation_dir = PROJECT_ROOT / "artifacts" / "presentation_demo"
presentation_dir.mkdir(parents=True, exist_ok=True)

presentation_ids = [
    "demo_purchase_01",
    "demo_purchase_02",
    "demo_stress_01",
    "demo_anger_01",
    "demo_neutral_01",
]

presentation_examples_df = examples_df.loc[examples_df["example_id"].isin(presentation_ids)].copy()
presentation_subset_path = presentation_dir / "presentation_subset.csv"
presentation_results_path = presentation_dir / "presentation_results.csv"

presentation_examples_df.to_csv(presentation_subset_path, index=False)

runner = EvaluationRunner(
    agent=agent,
    config=EvaluationRunnerConfig(
        dataset_path=presentation_subset_path,
        output_path=presentation_results_path,
        baseline_prompt_profile=BASELINE_PROMPT_PROFILE,
    ),
)

results = runner.run()
results_df = pd.read_csv(presentation_results_path)
display(
    results_df[[
        "example_id",
        "prediction_label",
        "risk_score",
        "protection_enabled",
        "baseline_prompt_profile",
        "guarded_response",
        "unguarded_response",
    ]]
)

print(f"Saved evaluation results to: {presentation_results_path}")